In [1]:
!pip install -q -U ultralytics
import ultralytics
print("Ultralytics version:", ultralytics.__version__)
!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 615.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics version: 8.4.138
Wed Sep  2 07:01:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf        

In [2]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        for f in files[:5]:
            print(f'{indent}  {f}')

input/
  datasets/
    alinoorqureshi/
      weapon-detection-yolo-optimized/
        dataset_merged/
          val/
            labels/
            images/
          test/
            labels/
            images/
          train/
            labels/
            images/


In [3]:
# Update this path based on what Cell 2 actually printed
WEAPON_YAML_ORIGINAL = "/kaggle/input/datasets/alinoorqureshi/weapon-detection-yolo-optimized/dataset_merged/data.yaml"

with open(WEAPON_YAML_ORIGINAL) as f:
    print(f.read())

names: [Weapon]
nc: 1
path: E:\FYP\MODEL_WORK\Weapons 2\dataset_merged
test: test/images
train: train/images
val: val/images



In [4]:
# Update BASE to match Cell 2's real folder structure
BASE = "/kaggle/input/datasets/alinoorqureshi/weapon-detection-yolo-optimized/dataset_merged"

# IMPORTANT: update nc and names below to match exactly what Cell 3 printed
corrected_weapon_yaml = f"""
train: {BASE}/train/images
val: {BASE}/val/images
test: {BASE}/test/images
nc: 2
names: ['gun', 'knife']
"""

with open("/kaggle/working/weapon_data.yaml", "w") as f:
    f.write(corrected_weapon_yaml)

WEAPON_DATA_YAML = "/kaggle/working/weapon_data.yaml"
print("Using corrected data.yaml:")
with open(WEAPON_DATA_YAML) as f:
    print(f.read())

Using corrected data.yaml:

train: /kaggle/input/datasets/alinoorqureshi/weapon-detection-yolo-optimized/dataset_merged/train/images
val: /kaggle/input/datasets/alinoorqureshi/weapon-detection-yolo-optimized/dataset_merged/val/images
test: /kaggle/input/datasets/alinoorqureshi/weapon-detection-yolo-optimized/dataset_merged/test/images
nc: 2
names: ['gun', 'knife']



In [5]:
from ultralytics import YOLO

weapon_model = YOLO("yolo11n.pt")

weapon_results = weapon_model.train(
    data=WEAPON_DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=32,
    device=[0, 1],
    project="/kaggle/working/runs",
    name="yolo11n_weapon",
)

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/weapon_data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo1

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30       3.5G      1.993      2.286       1.83         40        640: 100% ━━━━━━━━━━━━ 568/568 3.7it/s 2:33
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 71/71 3.5it/s 20.0s
                   all       4544       5935      0.449      0.312      0.309      0.135

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30       3.5G      1.936      2.169      1.787         31        640: 100% ━━━━━━━━━━━━ 568/568 3.8it/s 2:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 71/71 3.6it/s 19.6s
                   all       4544       5935      0.375       0.26      0.229     0.0956

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30       3.5G       1.91      2.107      1.771         28        640: 100% ━━━━━━━━━

In [6]:
weapon_best = "/kaggle/working/runs/yolo11n_weapon/weights/best.pt"
weapon_val_model = YOLO(weapon_best)
weapon_metrics = weapon_val_model.val(data=WEAPON_DATA_YAML)

print("YOLOv11n Weapon — mAP50-95:", weapon_metrics.box.map)
print("YOLOv11n Weapon — mAP50:", weapon_metrics.box.map50)
print("Precision:", weapon_metrics.box.mp)
print("Recall:", weapon_metrics.box.mr)

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 136.1±96.9 MB/s, size: 71.5 KB)
val: Scanning /kaggle/input/datasets/alinoorqureshi/weapon-detection-yolo-optimized/dataset_merged/val/labels... 4546 images, 497 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 4546/4546 1.2Kit/s 3.9s
val: /kaggle/input/datasets/alinoorqureshi/weapon-detection-yolo-optimized/dataset_merged/val/images/ds2_val_00034.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /kaggle/input/datasets/alinoorqureshi/weapon-detection-yolo-optimized/dataset_merged/val/images/ds2_val_00132.jpg: ignoring corrupt image/label: labels mix segment and detection rows
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/alinoorqureshi/weapon-detection-yolo-optimized/dataset_merged/val is not writable, cache not saved.
WARNING ⚠️ Box and segme

In [7]:
import glob, time

val_images = glob.glob(f"{BASE}/val/images/*.jpg")[:50]

start = time.time()
for img_path in val_images:
    _ = weapon_val_model.predict(img_path, verbose=False)
infer_ms = ((time.time() - start) / len(val_images)) * 1000
print(f"YOLOv11n Weapon average inference time: {infer_ms:.2f} ms/frame")

YOLOv11n Weapon average inference time: 19.40 ms/frame


In [8]:
import shutil
shutil.make_archive("/kaggle/working/yolo11n_weapon_results", 'zip', "/kaggle/working/runs/yolo11n_weapon")

from IPython.display import FileLink
FileLink("/kaggle/working/yolo11n_weapon_results.zip")

/kaggle/working/yolo11n_weapon_results.zip